# STEP 2 — 딥러닝 시장 레짐 예측 (S&P500)

HMM이 만든 향후 20일 레짐 라벨을 **과거 데이터만으로 예측**하는 Two-Track 모델을 학습하고,
단순 분류 정확도가 아니라 **배포가 정당화될 수준의 금융 검증**까지 수행한다.

**설계 골격(기획안 고정)**: Track1 GRU(60×12) + Track2 MLP(6) → Attention Late-Fusion →
Softmax(5). 분할 1999–2018 / 2019–2021 / 2022+. 손실 = class-weighted CrossEntropy
(+ 선택적 HMM 사후확률 distillation). Adam / ReduceLROnPlateau / EarlyStopping.

**핵심 가치 가설**: 강점은 초과수익이 아니라 **하락 방어**. 따라서 평가는 MDD·Calmar·Sortino 중심,
백테스트는 방어형 롱/현금 디리스킹(메인) + 확률가중(비교).

> 모듈(`config/dataset/model/train/evaluate/backtest/baselines.py`)을 `Deep/code/`에서 import 한다.

In [ ]:
# --- setup ---
import sys, os, glob, pathlib

# CODE_DIR 자동 탐색: 로컬/Colab 업로드/Drive 마운트 모두 대응
def _find_code_dir():
    candidates = [os.getcwd()]
    candidates += ["/content/Deep/code", "/content/code", "/content"]
    candidates += glob.glob("/content/drive/**/Deep/code", recursive=True)[:3]
    for p in glob.glob("/content/**/config.py", recursive=True):
        candidates.append(str(pathlib.Path(p).parent))
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.py")):
            return c
    return None

CODE_DIR = _find_code_dir()
if CODE_DIR is None:
    raise RuntimeError(
        "config.py를 찾을 수 없습니다.\n"
        "Colab 왼쪽 파일탐색기에서 config.py 위치를 확인하고\n"
        "아래 한 줄을 셀 맨 위에 추가하세요:\n"
        "  CODE_DIR = '/실제/경로/Deep/code'")

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
print("CODE_DIR:", CODE_DIR)

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
import config as C
from dataset import prepare_data, make_dataloaders
from model import RegimePredictor, count_params
from train import train_model, save_artifacts, set_seed
import evaluate as E, backtest as B, baselines as BL

set_seed(C.SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE)

## 1. 데이터 로드 · 분할 · 누수 방어
라벨은 미래 20일을 보므로 분할 경계마다 20거래일 embargo를 적용한다(코드 내 assert로 검증).

In [ ]:
data = prepare_data()
for name, s in [("train", data.train), ("val", data.val), ("test", data.test)]:
    print(f"{name:5s} n={len(s.y):5d}  X1={s.X1.shape}  X2={s.X2.shape}  "
          f"[{s.dates.min().date()} .. {s.dates.max().date()}]")
print("class_weights (Bear..Bull):", np.round(data.class_weights, 3))
dist = np.bincount(data.train.y, minlength=5)
print("train label dist:", dict(zip([C.REGIME_NAMES[i] for i in range(5)], dist)))
train_loader, val_loader, test_loader = make_dataloaders(data, batch_size=64)

## 2. 모델
GRU(64,2층)→32, MLP(6→32→16)→32, 두 모달리티 토큰에 attention(softmax) 가중합 → head→logits(5).

In [ ]:
m0 = RegimePredictor()
print(m0)
print("trainable params:", count_params(m0))

## 3. 학습 (+ 하이퍼파라미터 탐색)
기획안 구조는 고정, 기획안이 열어둔 값(lr·dropout·soft_weight)만 작은 그리드로 탐색한다.
탐색한 설정 수(`n_trials`)와 설정별 Sharpe 분산은 뒤의 **Deflated Sharpe**(다중검정 보정)에 사용된다.
`RUN_SWEEP=False`로 두면 기본 설정 1개만 학습.

In [ ]:
RUN_SWEEP = True
EPOCHS = 150

def val_sharpe(model, temp):
    probs, preds, _ = E.collect_predictions(model, val_loader, DEVICE, temp)
    bt = B.run_backtest(data.val.dates, data.val.close, preds, probs, data.val.y)
    return B.perf_metrics(bt["pnl::Defensive (long/cash)"].values)["Sharpe"]

if RUN_SWEEP:
    grid = [dict(lr=lr, dropout=do, soft_weight=sw)
            for lr in (1e-3, 3e-4) for do in (0.3, 0.4) for sw in (0.0, 0.3)]
else:
    grid = [dict(lr=1e-3, dropout=0.3, soft_weight=0.3)]

results, best = [], None
for hp in grid:
    model, hist, temp, dev = train_model(data, max_epochs=EPOCHS, **hp, verbose=False)
    probs, preds, yv = E.collect_predictions(model, val_loader, DEVICE, temp)
    f1 = E.classification_metrics(yv, preds, verbose=False)["macro_f1"]
    sh = val_sharpe(model, temp)
    results.append({**hp, "val_macroF1": f1, "val_Sharpe": sh})
    if best is None or f1 > best["f1"]:
        best = dict(model=model, hist=hist, temp=temp, hp=hp, f1=f1)
    print(f"{hp}  val_F1={f1:.3f}  val_Sharpe={sh:.2f}")

res_df = pd.DataFrame(results)
N_TRIALS = len(grid)
SR_TRIALS_STD = float(res_df["val_Sharpe"].std(ddof=1)) if len(res_df) > 1 else 0.4
print("\nBest:", best["hp"], "val_macroF1=%.3f" % best["f1"],
      "| N_TRIALS=%d  SR_std=%.3f" % (N_TRIALS, SR_TRIALS_STD))
model, history, temperature = best["model"], best["hist"], best["temp"]

In [ ]:
# 학습 곡선
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_macro_f1, color="green"); ax[1].set_title("val macro-F1")
plt.tight_layout(); plt.show()
print("temperature (calibration):", round(temperature, 3))

In [ ]:
save_artifacts(model, data, temperature, best["hp"])

## 4. 분류 · 캘리브레이션 평가
하락 포착(Bear/Weak Bear recall, risk-off AUROC)과 확률 신뢰도(ECE/Brier, 온도보정 전후)를 본다.

In [ ]:
probs_raw, preds_raw, y_test = E.collect_predictions(model, test_loader, DEVICE, temperature=1.0)
probs, preds, _              = E.collect_predictions(model, test_loader, DEVICE, temperature=temperature)
report = E.full_report(y_test, probs, preds, label="DL (calibrated)")
print("\nECE 보정 전:", round(E.calibration_metrics(y_test, probs_raw)["ECE"], 3),
      "→ 보정 후:", round(report["ECE"], 3))

In [ ]:
# 혼동행렬 + 신뢰도 다이어그램
names = [C.REGIME_NAMES[i] for i in range(5)]
cm = report["confusion_matrix"]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(5)); ax[0].set_xticklabels(names, rotation=45, ha="right")
ax[0].set_yticks(range(5)); ax[0].set_yticklabels(names)
for i in range(5):
    for j in range(5):
        ax[0].text(j, i, cm[i, j], ha="center", va="center",
                   color="white" if cm[i, j] > cm.max()/2 else "black")
ax[0].set_title("Confusion (true→pred)"); ax[0].set_ylabel("true")
for p, lab in [(probs_raw, "raw"), (probs, "calibrated")]:
    xs, ys = E.reliability_curve(y_test, p)
    ax[1].plot(xs, ys, marker="o", label=lab)
ax[1].plot([0, 1], [0, 1], "k--", lw=1)
ax[1].set_xlabel("confidence"); ax[1].set_ylabel("accuracy")
ax[1].set_title("Reliability"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. 금융 백테스트 (방어 우선)
신호=종가 t → **익일 체결**, 거래비용 5bps/회전, 현금 0%. 전략: 방어형 롱/현금(메인) + 확률가중(비교),
벤치마크: Buy&Hold, Oracle(미래라벨 사용 상한). 메인 지표는 MDD·Calmar·Sortino.

In [ ]:
bt = B.run_backtest(data.test.dates, data.test.close, preds, probs, y_test)
summary = B.summarize(bt, n_trials=N_TRIALS, sr_trials_std=SR_TRIALS_STD)
print(summary.round(3).to_string())

In [ ]:
# 누적수익 + Drawdown
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})
for name in ["Defensive (long/cash)", "Prob-weighted", "Buy&Hold", "Oracle"]:
    eq = (1 + bt[f"pnl::{name}"]).cumprod()
    ax[0].plot(bt.index, eq, label=name)
    if name in ("Defensive (long/cash)", "Buy&Hold"):
        dd = eq / eq.cummax() - 1
        ax[1].fill_between(bt.index, dd, 0, alpha=0.4, label=name)
ax[0].set_yscale("log"); ax[0].set_title("Equity (log)"); ax[0].legend()
ax[1].set_title("Drawdown"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 하위기간 스트레스 (테스트 구간: 2022 약세장 / 2023-24 강세장)
print(B.subperiod_analysis(bt, B.SUBPERIODS).round(3).to_string())
print("\n참고: COVID 2020은 학습구간 안에 있어 테스트 백테스트에 없음 → 7번 walk-forward에서 OOS로 검증.")

## 6. 베이스라인 비교 (딥러닝 복잡도 정당화)
DL이 risk-off AUROC와 백테스트 Calmar에서 단순 모델을 이겨야 한다.

In [ ]:
rows = {}
def metrics_row(preds_, probs_):
    cls = E.classification_metrics(y_test, preds_, verbose=False)
    auc = E.risk_off_auroc(y_test, probs_)
    bb = B.run_backtest(data.test.dates, data.test.close, preds_, probs_, y_test)
    pm = B.perf_metrics(bb["pnl::Defensive (long/cash)"].values,
                        bb["w::Defensive (long/cash)"].values)
    return {"macro_F1": cls["macro_f1"], "risk_off_AUROC": auc["risk_off_auroc"],
            "Calmar": pm["Calmar"], "MDD": pm["MDD"], "Sharpe": pm["Sharpe"]}

rows["DL (ours)"] = metrics_row(preds, probs)
for name in ["majority", "lagged", "logistic", "xgboost"]:
    try:
        bp, bpr = BL.ALL[name](data)
        rows[name] = metrics_row(bp, bpr)
    except Exception as e:
        print(f"[skip {name}] {e}")
print(pd.DataFrame(rows).T.round(3).to_string())

## 7. Walk-Forward (확장창) 검증 — 실제 배포 시뮬레이션
매년 직전 데이터로 재학습 후 그 해를 OOS 예측. 2020년 폴드로 **COVID 급락을 OOS로 스트레스**한다.
**시드 앙상블**(여러 시드로 학습 후 확률 평균)으로 단일 폴드가 한 클래스로 붕괴하는 불안정을 완화하고,
폴드별 **risk-off AUROC**로 "방향을 맞췄는지(실력)"를 macro-F1과 분리해 본다.
(폴드 × 시드 만큼 재학습 → 시간 소요. `WF_SEEDS`·폴드 수로 조절하라.)

In [ ]:
WF_YEARS = [2020, 2022, 2023, 2024]   # 필요시 [2018..2025] 확장
WF_SEEDS = [0, 1, 2]                   # 시드 앙상블: 폴드별 붕괴(한 클래스 쏠림) 완화
wf_frames, wf_diag = [], []
for yr in WF_YEARS:
    d_wf = prepare_data(train_end=f"{yr-2}-12-31", val_start=f"{yr-1}-01-01",
                        test_start=f"{yr}-01-01", test_end=f"{yr}-12-31")
    tl_w, vl_w, tel_w = make_dataloaders(d_wf, 64)
    # --- 시드 앙상블: 여러 시드로 학습 후 확률(temperature 적용)을 평균 ---
    prob_stack, y_w, dev_w = [], None, None
    for sd in WF_SEEDS:
        mdl_w, _, tmp_w, dev_w = train_model(d_wf, max_epochs=120, seed=sd,
                                             **best["hp"], verbose=False)
        p_w, _, y_w = E.collect_predictions(mdl_w, tel_w, dev_w, tmp_w)
        prob_stack.append(p_w)
    probs_w = np.mean(prob_stack, axis=0)
    preds_w = probs_w.argmax(1)
    wf_frames.append(B.run_backtest(d_wf.test.dates, d_wf.test.close, preds_w, probs_w, y_w))
    f1 = E.classification_metrics(y_w, preds_w, verbose=False)["macro_f1"]
    try:
        auc = E.risk_off_auroc(y_w, probs_w)["risk_off_auroc"]
    except ValueError:
        auc = float("nan")   # 폴드에 risk-off 라벨이 없으면 AUROC 미정의
    bc = np.bincount(preds_w, minlength=5)
    wf_diag.append({"year": yr, "n": len(y_w), "macroF1": round(f1, 3),
                    "risk_off_AUROC": round(auc, 3), "preds": bc.tolist()})
    print(f"WF {yr}: n={len(y_w)}  macroF1={f1:.3f}  risk_off_AUROC={auc:.3f}  preds={bc}")

wf = pd.concat(wf_frames).sort_index()
print("\n=== Walk-forward OOS 종합 (시드 앙상블) ===")
print(B.summarize(wf, n_trials=N_TRIALS, sr_trials_std=SR_TRIALS_STD).round(3).to_string())
covid = wf.loc["2020-02-01":"2020-04-30"]
if len(covid):
    dmdd = B.perf_metrics(covid["pnl::Defensive (long/cash)"].values)["MDD"]
    bmdd = B.perf_metrics(covid["pnl::Buy&Hold"].values)["MDD"]
    print(f"\nCOVID(OOS) MDD  방어={dmdd:.3f}  vs  Buy&Hold={bmdd:.3f}")

## 9. 배포 인터페이스 (다음 단계)
`artifacts/`에 `ensemble_seed{0..k}.pt`, `scaler_t1/t2.pkl`, `config.json`(시드별 온도·모델구조·피처·사다리)이 저장됨.

**배포(STEP 3)**: 라이브 SPY(FinanceDataReader)+FRED ALFRED first_release(발표지연 offset, STEP1 재사용)
→ 동일 피처 생성 → 저장 스케일러 transform → 앙상블 확률 평균 → 레짐 확률 + 권장 익스포저.
`serve.py`(추론·개인화 사다리) + `streamlit_app.py`(대시보드)로 구현. 개인화는 모델 이후 단계(같은 확률 × 사용자별 사다리)에서만 처리.

**합격 기준**: test에서 Buy&Hold 대비 MDD·Calmar 유의 개선 + risk-off AUROC ≳ 0.80 + DSR 유의 +
DL이 베이스라인 상회 + walk-forward(특히 2020 OOS)에서 방어 유지.

In [ ]:
# 배포용 시드 앙상블 학습 (표준 분할: train<=2018 / val 2019-21 / test 2022+)
# -> 2022+ 테스트가 그대로 OOS로 유지되므로 저장된 앙상블을 그대로 검증할 수 있다.
import joblib, json
ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]

ens_state, ens_temps, prob_stack = [], [], []
for sd in ENSEMBLE_SEEDS:
    mdl, _, tmp, _ = train_model(data, max_epochs=EPOCHS, seed=sd, **best["hp"], verbose=False)
    p, _, _ = E.collect_predictions(mdl, test_loader, DEVICE, tmp)
    ens_state.append({k: v.cpu().clone() for k, v in mdl.state_dict().items()})
    ens_temps.append(tmp); prob_stack.append(p)

probs_ens = np.mean(prob_stack, axis=0)        # 시드별 (온도보정) 확률 평균
preds_ens = probs_ens.argmax(1)
print("ensemble preds dist:", np.bincount(preds_ens, minlength=5))
print("ensemble macro-F1 :", round(E.classification_metrics(y_test, preds_ens, verbose=False)["macro_f1"], 3))
print("ensemble risk-off AUROC:", round(E.risk_off_auroc(y_test, probs_ens)["risk_off_auroc"], 3))

# --- 저장: 시드별 가중치 + 스케일러 + config.json (serve.py가 읽음) ---
C.ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
for sd, st in zip(ENSEMBLE_SEEDS, ens_state):
    torch.save(st, C.ARTIFACT_DIR / f"ensemble_seed{sd}.pt")
joblib.dump(data.scaler_t1, C.ARTIFACT_DIR / "scaler_t1.pkl")
joblib.dump(data.scaler_t2, C.ARTIFACT_DIR / "scaler_t2.pkl")
meta = {
    "track1_feats": C.TRACK1_FEATS, "track2_feats": C.TRACK2_FEATS,
    "seq_len": C.SEQ_LEN, "horizon": C.HORIZON, "n_regimes": C.N_REGIMES,
    "regime_names": {str(k): v for k, v in C.REGIME_NAMES.items()},
    "risk_off_regimes": list(C.RISK_OFF_REGIMES),
    "exposure_ladder": {str(k): v for k, v in C.EXPOSURE_LADDER.items()},
    "model_kwargs": {"gru_hidden": 64, "gru_layers": 2, "fusion_dim": 32,
                     "dropout": float(best["hp"].get("dropout", 0.3))},
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "temperatures": {str(sd): float(t) for sd, t in zip(ENSEMBLE_SEEDS, ens_temps)},
    "class_weights": data.class_weights.tolist(),
    "hparams": best["hp"],
    "splits": {"train_end": C.TRAIN_END, "val": [C.VAL_START, C.VAL_END],
               "test_start": C.TEST_START},
}
(C.ARTIFACT_DIR / "config.json").write_text(json.dumps(meta, indent=2, ensure_ascii=False))
print("saved ensemble (%d seeds) ->" % len(ENSEMBLE_SEEDS), C.ARTIFACT_DIR)

## 8. 저장물 & 배포 인터페이스 (다음 단계)
`artifacts/`에 `best_model.pt`, `scaler_t1/t2.pkl`, `config.json`(피처·온도·하이퍼파라미터) 저장됨.

**배포(다음 STEP)**: 라이브 SPY(FinanceDataReader)+FRED ALFRED first_release(발표지연 offset, STEP1 재사용)
→ 동일 피처 생성 → 저장 스케일러 transform → 모델 → 레짐 확률 + 권장 익스포저. Streamlit 대시보드.

**합격 기준**: test에서 Buy&Hold 대비 MDD·Calmar 유의 개선 + risk-off AUROC ≳ 0.80 + DSR 유의 +
DL이 베이스라인 상회 + walk-forward(특히 2020 OOS)에서 방어 유지.